# WeedsGalore Dataset Exploration
Visualize samples, class distributions, and band statistics.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import yaml
from src.data.dataset import WeedsGaloreDataset
from src.data.transforms import get_val_transforms
from src.utils.visualization import mask_to_color, CLASS_NAMES

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
ds_cfg = cfg['dataset']
print('Config loaded')

In [ ]:
# Load dataset
transform = get_val_transforms(512, cfg.get('augmentation', {}))
train_ds = WeedsGaloreDataset(
    root_dir='../data/raw/weedsgalore-dataset',
    split='train',
    image_size=512,
    transform=transform,
)
print(train_ds)

In [ ]:
# Visualize 6 samples
fig, axes = plt.subplots(6, 3, figsize=(12, 18))
for i in range(6):
    sample = train_ds[i * 10]
    img = sample['image'].permute(1,2,0).numpy()
    # Denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = (img * std + mean).clip(0, 1)
    mask = sample['mask'].numpy()
    color_mask = mask_to_color(mask)

    axes[i,0].imshow(img);              axes[i,0].set_title(f"Image [{sample['stem']}]")
    axes[i,1].imshow(color_mask);       axes[i,1].set_title('Semantic Mask')
    blend = (img * 255 * 0.6 + color_mask * 0.4).clip(0,255).astype('uint8')
    axes[i,2].imshow(blend);            axes[i,2].set_title('Overlay')
    for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.savefig('samples.png', dpi=120)
plt.show()

In [ ]:
# Class distribution
import torch
counts = torch.zeros(3)
for i in range(min(200, len(train_ds))):
    mask = train_ds[i]['mask']
    for c in range(3):
        counts[c] += (mask == c).sum()

total = counts.sum()
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(CLASS_NAMES, (counts / total * 100).numpy(), color=['#555','#2a9d4e','#d62828'])
ax.set_ylabel('Pixel %')
ax.set_title('Class pixel distribution (first 200 samples)')
for j, v in enumerate((counts / total * 100).numpy()):
    ax.text(j, v + 0.5, f'{v:.1f}%', ha='center')
plt.tight_layout()
plt.show()
print({n: f'{p:.1f}%' for n, p in zip(CLASS_NAMES, (counts/total*100).numpy())})

In [ ]:
# Band statistics (R, G, B channel mean/std)
n = min(100, len(train_ds))
means = []
for i in range(n):
    img = train_ds[i]['image']  # (3, H, W) float, already normalized
    means.append(img.mean(dim=(1,2)).numpy())
means = np.array(means)
print('Per-channel mean (normalized):',  means.mean(axis=0).round(4))
print('Per-channel std  (normalized):',  means.std(axis=0).round(4))